# PWKD Option 1 — Pretrained ResNet18 Student

Implements **Pruning While Knowledge Distillation** (Wang et al., 2025) using:
- **Teacher**: frozen pretrained ResNet50 (RadImageNet weights)
- **Student**: pretrained ResNet18 (RadImageNet weights), trained from a
  warm starting point rather than random initialisation

This is a cross-architecture setup. The student is smaller by design
(ResNet18 vs ResNet50), and PWKD simultaneously prunes it further and
distils knowledge from the teacher.

Produces 5 compressed models at pruning ratios [10%, 25%, 50%, 70%, 90%],
saved to `trained_models/pwkd_r18_<ratio>/` and uploaded to HuggingFace.

**Run all cells top to bottom. Requires GPU.**

In [1]:
import os, copy, subprocess
import torch

# navigate to project root
target = 'CS6423_knowledge_distillation_project'
if not os.getcwd().endswith(target):
    import sys
    os.chdir(os.path.join(os.getcwd(), target))
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f'Working dir: {os.getcwd()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

subprocess.run(['pip', 'install', 'PyWavelets', '--quiet'], check=True)

Working dir: /home/cor10/CS6423_knowledge_distillation_project
Device: cuda


CompletedProcess(args=['pip', 'install', 'PyWavelets', '--quiet'], returncode=0)

In [2]:
import pandas as pd
from modules.dataset_prepper import datasetPrepper

data_prep = datasetPrepper(
    dataframe_path='data/labels.csv',
    image_dir='data/test_images',
).prepare(compute_class_weights=True)

NUM_CLASSES = len(data_prep.class_names)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(data_prep.train_loader)} | Val batches: {len(data_prep.val_loader)}')

Classes: 61
Train batches: 249 | Val batches: 50


In [3]:
from modules.imagenet_loader import ImagenetLoader
from modules.evaluate_model import ModelEvaluator

loader = ImagenetLoader()

# Teacher: frozen pretrained ResNet50
teacher = loader.load_radimagenet_resnet50(
    weights_path='trained_models/resnet50_baseline_gpu_new/resnet50_baseline_gpu_new.pth',
    load_type='load'
)
teacher = teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad = False

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

teacher_metrics = evaluator.evaluate_single(teacher, 'ResNet50_teacher')
print(f'Teacher F1: {teacher_metrics["f1_macro"]:.4f}')
print(f'Teacher params: {teacher_metrics["total_parameters"]:,}')


Warming up ResNet50_teacher...
Running inference...
Teacher F1: 0.4867
Teacher params: 23,633,021


In [4]:
# Student baseline: pretrained ResNet18 before any PWKD
# This gives us a warm starting point (F1 ~ 0.4) rather than random init
loader.load_radimagenet_resnet18(
    weights_path='trained_models/resnet18_baseline_gpu_new/resnet18_baseline_gpu_new.pth',
    load_type='load'
)
# loader.freeze_backbone()
student_base = loader.model
student_base = student_base.to(device)

baseline_metrics = evaluator.evaluate_single(student_base, 'ResNet18_pretrained_baseline')
BASELINE_PARAMS  = baseline_metrics['total_parameters']
print(f'ResNet18 pretrained baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 pretrained baseline params: {BASELINE_PARAMS:,}')


Warming up ResNet18_pretrained_baseline...
Running inference...
ResNet18 pretrained baseline F1:    0.4054
ResNet18 pretrained baseline params: 11,207,805


## Channel count reference

ResNet18 and ResNet50 have different channel widths at each stage, so the
wavelet alignment modules need a 1×1 projection to match teacher → student:

| Stage   | ResNet50 (teacher) | ResNet18 (student) |
|---------|-------------------|-------------------|
| layer2  | 512               | 128               |
| layer3  | 1024              | 256               |
| layer4  | 2048              | 512               |

In [5]:
import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
# from pwkd import PWKDLoss, make_aux_fn, finalise_student
import importlib, pwkd
# importlib.reload(pwkd)
from pwkd.pwkd import PWKDLoss, make_aux_fn, finalise_student

TEACHER_CHANNELS = {'layer2': 512,  'layer3': 1024, 'layer4': 2048}
STUDENT_CHANNELS = {'layer2': 128,  'layer3': 256,  'layer4': 512}

PRUNING_RATIOS = [0, 0.05, 0.10, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
# LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 5e-4

LAM            = 0.5    # increase KD weight so teacher signal is stronger
# SPARSE_WEIGHT  = 0.0    # 0 ratio so no point having sparsity penalty
learn_rate     = 1e-4   # much lower — fine-tuning a pretrained model

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)




pwkd_metrics = []

print(f'Teacher F1: {teacher_metrics["f1_macro"]:.4f}')
print(f'Teacher params: {teacher_metrics["total_parameters"]:,}')

print(f'ResNet18 pretrained baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 pretrained baseline params: {BASELINE_PARAMS:,}')


for ratio in PRUNING_RATIOS:
    label = f'pwkd_r18_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 1 (ResNet18) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    # Each ratio starts from the same pretrained weights
    student = copy.deepcopy(student_base).to(device)
    for p in student.parameters():
        p.requires_grad = True
        
    
    # reset teacher each experiment
    teacher_copy = copy.deepcopy(teacher).to(device)
    teacher_copy.eval()
    

        
    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher_copy,
        pruning_ratio    = ratio,
        teacher_channels = TEACHER_CHANNELS,
        student_channels = STUDENT_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = 0 if ratio == 0 else SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        # learn_rate = 5e-4,
        learn_rate = learn_rate,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=learn_rate, weight_decay=1e-4,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher_copy), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    # Save
    import os
    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    # Evaluate
    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,} | '
          f'latency: {metrics["avg_latency_ms"]:.2f}ms')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
print('\nPWKD Option 1 (ResNet18) — Results')
display(summary_df)


Teacher F1: 0.4867
Teacher params: 23,633,021
ResNet18 pretrained baseline F1:    0.4054
ResNet18 pretrained baseline params: 11,207,805

  PWKD Option 1 (ResNet18) — pruning ratio 0%


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.11batch/s]



Epoch 1/10
Train Loss: 1.3073 | Train F1: 0.6374
Val Loss: 1.5710 | Val F1: 0.4002
Epoch Time: 37.34s



Validating: 100%|██████████| 50/50 [00:02<00:00, 18.09batch/s]



Epoch 2/10
Train Loss: 0.9745 | Train F1: 0.7004
Val Loss: 1.5115 | Val F1: 0.4071
Epoch Time: 37.65s



...


Epoch 9/10
Train Loss: 0.8210 | Train F1: 0.7977
Val Loss: 1.5137 | Val F1: 0.4201
Epoch Time: 37.85s



Validating: 100%|██████████| 50/50 [00:02<00:00, 18.04batch/s]


Epoch 10/10
Train Loss: 0.8109 | Train F1: 0.7969
Val Loss: 1.4725 | Val F1: 0.4380
Epoch Time: 37.73s



Finalised: 1724/1920 conv1 channels zeroed (89.8%)
Saved → trained_models/pwkd_r18_r90/pwkd_r18_r90_full.pth

Warming up pwkd_r18_r90...
Running inference...
  ratio 90% | F1: 0.0003 | params: 1,339,197 | latency: 0.39ms

PWKD Option 1 (ResNet18) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
0%,0.27,0.4250,42.8,0.39
5%,4.74,0.3930,42.8,0.39
10%,9.69,0.3673,42.8,0.39
15%,14.54,0.3140,42.8,0.39
20%,19.49,0.2998,42.8,0.40
25%,24.46,0.2371,42.8,0.39
30%,29.24,0.0893,42.8,0.42
35%,34.20,0.0176,42.8,0.78
40%,39.04,0.0104,42.8,1.09


<!-- # only run this cell if not running the previous cell

import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
# from pwkd import PWKDLoss, make_aux_fn, finalise_student
import importlib, pwkd
importlib.reload(pwkd)
from pwkd import PWKDLoss, make_aux_fn, finalise_student

TEACHER_CHANNELS = {'layer2': 512,  'layer3': 1024, 'layer4': 2048}
STUDENT_CHANNELS = {'layer2': 128,  'layer3': 256,  'layer4': 512}

PRUNING_RATIOS = [0.10, 0.25, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)




pwkd_metrics = [] -->

In [12]:
# # ── Extra ratios to better capture the Pareto curve descent ──────────────────
# # Run this cell independently — existing trained models are not affected.

# EXTRA_RATIOS = [0.2, 0.3, 0.35, 0.40, 0.45]   # between the good and collapsed points

# # for ratio in EXTRA_RATIOS:
#     label = f'pwkd_r18_r{int(ratio * 100)}'
#     print(f'\n{"="*60}')
#     print(f'  PWKD Option 1 (ResNet18) — pruning ratio {ratio:.0%}')
#     print(f'{"="*60}')

#     student = copy.deepcopy(student_base).to(device)
#     for p in student.parameters():
#         p.requires_grad = True

#     pwkd_loss = PWKDLoss(
#         student          = student,
#         teacher          = teacher,
#         pruning_ratio    = ratio,
#         teacher_channels = TEACHER_CHANNELS,
#         student_channels = STUDENT_CHANNELS,
#         class_weights    = data_prep.class_weights.to(device)
#                            if data_prep.class_weights is not None else None,
#         lam              = LAM,
#         kd_temp          = KD_TEMP,
#         sparse_weight    = SPARSE_WEIGHT,
#     ).to(device)

#     trainer = modelTrainer(
#         model      = student,
#         data_prep  = data_prep,
#         device     = device,
#         learn_rate = 5e-4,
#         num_epochs = NUM_EPOCHS,
#         model_name = label,
#     )
#     trainer.loss_fn   = pwkd_loss
#     trainer.optimizer = torch.optim.AdamW(
#         list(student.parameters()) + list(pwkd_loss.parameters()),
#         lr=5e-4, weight_decay=1e-2,
#     )
#     trainer.create_classnum_to_label_map(data_prep.class_names)
#     trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

#     student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

#     save_dir = os.path.join('trained_models', label)
#     os.makedirs(save_dir, exist_ok=True)
#     torch.save({'model': student, 'epoch': NUM_EPOCHS},
#                os.path.join(save_dir, f'{label}_full.pth'))
#     print(f'Saved → {save_dir}/{label}_full.pth')

#     student.eval()
#     metrics = evaluator.evaluate_single(student, label)

#     pwkd_metrics.append({
#         'Pruning Ratio':      f'{int(ratio*100)}%',
#         'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
#                                     / BASELINE_PARAMS * 100, 2),
#         'F1 Score':           round(metrics['f1_macro'],     4),
#         'Size (MB)':          round(metrics['model_size_mb'], 1),
#         'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
#     })
#     print(f'  ratio {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
#           f'params: {metrics["total_parameters"]:,}')

# summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
# display(summary_df)

In [6]:
print('\nPWKD Option 1 (ResNet18) — Results')
display(summary_df)


PWKD Option 1 (ResNet18) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
0%,0.27,0.4250,42.8,0.39
5%,4.74,0.3930,42.8,0.39
10%,9.69,0.3673,42.8,0.39
15%,14.54,0.3140,42.8,0.39
20%,19.49,0.2998,42.8,0.40
25%,24.46,0.2371,42.8,0.39
30%,29.24,0.0893,42.8,0.42
35%,34.20,0.0176,42.8,0.78
40%,39.04,0.0104,42.8,1.09
